# Import Library 

In [1]:
import pandas as pd
import numpy as np
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import os
import string as st

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

In [3]:
str(sorted([154, 184, 77, 89, 82, 187, 10, 38, 39, 93, 66, 90, 50, 36, 32, 30, 40, 34, 147, 198, 120, 150, 200, 7, 67, 48, 110, 61, 191, 45, 149, 132, 127, 103, 64, 47, 181, 65, 111, 175, 193, 152, 182, 188]))

'[7, 10, 30, 32, 34, 36, 38, 39, 40, 45, 47, 48, 50, 61, 64, 65, 66, 67, 77, 82, 89, 90, 93, 103, 110, 111, 120, 127, 132, 147, 149, 150, 152, 154, 175, 181, 182, 184, 187, 188, 191, 193, 198, 200]'

# Functions

In [2]:
def preprocess_text(text):
    # Define interrogative words to KEEP
    interrogatives = {"what", "why", "how", "who", "where", "when", "which", "whom", "whose", "no", "not",
                    "very" ,"too" ,"too" ,"just", "if", "but", "however", "without", "like"}
    custom_stopwords = set(nlp.Defaults.stop_words)
    custom_stopwords -= interrogatives

    doc = nlp(text.lower().strip())  # Lowercase and remove whitespace
    
# Process tokens: lemmatize, filter stopwords/punct/numbers, keep interrogatives
    tokens = [
        token.lemma_ 
        for token in doc 
        if (
            (not token.is_stop or token.text in interrogatives) and  # Keep interrogatives
            not token.is_punct and token.is_alpha                                  # Remove punctuation
            # (token.is_alpha or token.like_num)                       # Keep words/numbers
        )
    ]

    return ' '.join(tokens)

# Import Dataset

In [3]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}
label_mapper = {
    'BT1' : 'knowledge',
    'BT2' : 'comprehension',
    'BT3' : 'application',
    'BT4' : 'analysis',
    'BT5' : 'synthesis',
    'BT6' : 'evaluation'
}


# Load dataset
df = pd.DataFrame()
for i in [2,3,4,5]:
    q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(i) + '.csv')
    q_df['dataset_id'] = i
    df = pd.concat([df , q_df], )
    
df = df.reset_index(drop=True)

# Apply preprocessing
df['label'] = df['label'].replace(label_mapper)
df['label'] = df['label'].str.lower()
df['label'] = df['label'].replace(mapping)

df['processed_question'] = df['question'].apply(preprocess_text)
df['processed_question'] = [''.join(text) for text in df['processed_question']]

print(df['label'].value_counts())

label
comprehension    861
knowledge        244
evaluation       218
application      216
analysis         204
synthesis        179
Name: count, dtype: int64


In [4]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}
label_mapper = {
    'BT1' : 'knowledge',
    'BT2' : 'comprehension',
    'BT3' : 'application',
    'BT4' : 'analysis',
    'BT5' : 'synthesis',
    'BT6' : 'evaluation'
}


# Load dataset
test_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(1) + '.csv')
    
test_df = test_df.reset_index(drop=True)

# Apply preprocessing
test_df['label'] = test_df['label'].replace(label_mapper)
test_df['label'] = test_df['label'].str.lower()
test_df['label'] = test_df['label'].replace(mapping)

test_df['processed_question'] = test_df['question'].apply(preprocess_text)
test_df['processed_question'] = [''.join(text) for text in test_df['processed_question']]
print(test_df['label'].value_counts())

label
analysis         100
knowledge        100
comprehension    100
evaluation       100
synthesis        100
application      100
Name: count, dtype: int64


# EDA

## OOV test

In [45]:
def get_vocab(text_series):
    vocab = set()
    for text in text_series.dropna():
        words = text.split()
        vocab.update(words)
    return vocab

In [46]:
label_oov_dict = {}

for label in test_df['label'].unique():
    train_texts = df[df['label'] == label]['processed_question']
    test_texts = test_df[test_df['label'] == label]['processed_question']
    train_vocab = get_vocab(train_texts)
    test_vocab = get_vocab(test_texts)
    oov_words = test_vocab - train_vocab
    label_oov_dict[label] = {
        'num_oov': len(oov_words),
        'oov_words': list(oov_words)
    }

In [47]:
for label, oov_info in label_oov_dict.items():
    print(f"Label: {label}")
    print(f"  Number of OOV words: {oov_info['num_oov']}")
    print(f"  Sample OOV words: {oov_info['oov_words']}")
    print("-" * 50)

Label: analysis
  Number of OOV words: 242
  Sample OOV words: ['paris', 'mainstream', 'sell', 'water', 'action', 'significant', 'attack', 'spring', 'marineris', 'series', 'heterodox', 'family', 'venn', 'contemporary', 'interact', 'trip', 'privilege', 'pot', 'interrogation', 'switch', 'make', 'iris', 'gps', 'feeling', 'where', 'camera', 'psychologist', 'rct', 'class', 'fact', 'ago', 'operator', 'buyer', 'narrative', 'pet', 'adolescent', 'preview', 'safe', 'coastal', 'reasonably', 'ethnic', 'conjugal', 'olympic', 'floodplain', 'place', 'attitude', 'society', 'z', 'position', 'purge', 'pointwise', 'dog', 'question', 'heme', 'male', 'asd', 'course', 'tectonic', 'td', 'endorsement', 'old', 'gun', 'carnivorous', 'positivist', 'undercover', 'rum', 'positive', 'fox', 'kuhn', 'influential', 'exclude', 'tdap', 'individual', 'man', 'paragraph', 'exercise', 'eat', 'food', 'trajectory', 'child', 'party', 'monthly', 'bulimia', 'effectively', 'investigation', 'hypothesis', 'prototyping', 'offender',

## Extract Bloom's Taxonomy Action Verb

In [48]:
bloom_verbs = {
    "knowledge": [
        "arrange", "choose", "cite", "copy", "define", "describe", "draw", "duplicate", "identify", "indicate",
        "label", "list", "locate", "match", "memorize", "name", "order", "outline", "quote", "read", "recall",
        "recite", "recognize", "record", "relate", "repeat", "reproduce", "review", "select", "state",
        "tabulate", "tell", "underline", "write"
    ],
    "comprehension": [
        "articulate", "associate", "characterize", "cite", "clarify", "classify", "compare", "contrast",
        "defend", "demonstrate", "describe", "differentiate", "discuss", "distinguish", "estimate", "explain",
        "extend", "extrapolate", "generalize", "give", "give examples", "identify", "illustrate", "indicate",
        "infer", "interpolate", "interpret", "locate", "match", "observe", "organize", "paraphrase", "predict",
        "recognize", "relate", "report", "represent", "restate", "review", "rewrite", "select", "summarize",
        "tell", "translate"
    ],
    "application": [
        "act", "adapt", "apply", "back/back up", "change", "classify", "complete", "compute", "construct",
        "demonstrate", "develop", "discover", "dramatize", "employ", "experiment", "explain", "generalize",
        "identify", "illustrate", "implement", "interpret", "interview", "manipulate", "modify", "operate",
        "organize", "paint", "practice", "predict", "prepare", "produce", "relate", "schedule", "select",
        "show", "sketch", "solve", "translate", "use", "utilize", "write"
    ],
    "analysis": [
        "analyze", "appraise", "break", "break down", "calculate", "categorize", "classify", "compare",
        "conclude", "contrast", "correlate", "criticize", "debate", "deduce", "detect", "diagnose", "diagram",
        "differentiate", "discover", "discriminate", "dissect", "distinguish", "divide", "evaluate", "examine",
        "experiment", "figure", "group", "identify", "illustrate", "infer", "inspect", "inventory", "investigate",
        "order", "organize", "outline", "point out", "predict", "prioritize", "question", "relate", "select",
        "separate", "solve", "subdivide", "test"
    ],
    "evaluation": [
        "appraise", "argue", "arrange", "assess", "attach", "choose", "compare", "conclude", "core", "criticize",
        "critique", "decide", "defend", "describe", "design", "determine", "estimate", "evaluate", "explain",
        "grade", "invent", "judge", "manage", "mediate", "prepare", "probe", "rate", "rearrange", "reconcile",
        "release", "rewrite", "select", "set up", "supervise", "synthesize", "test", "value", "verify", "weigh"
    ],
    "synthesis": [
        "arrange", "assemble", "categorize", "choose", "collect", "combine", "compile", "compose", "construct",
        "create", "design", "develop", "devise", "estimate", "evaluate", "explain", "facilitate", "formulate",
        "generate", "hypothesize", "improve", "integrate", "invent", "make", "manage", "modify", "organize",
        "originate", "plan", "predict", "produce", "propose", "rate", "rearrange", "reconstruct", "relate",
        "reorganize", "revise", "rewrite", "role-play", "set up", "specify", "summarize", "synthesize",
        "tell/tell why", "write"
    ]
}


In [50]:
# lowercase all oov words for matching
oov_words_set = set(word.lower() for word in oov_words)  # from your earlier extraction

level_oov_matches = {'knowledge':[],
                     'comprehension':[],
                     'application':[],
                     'analysis':[],
                     'synthesis':[],
                     'evaluation':[]}

for level in bloom_verbs.keys():
    for verb in label_oov_dict[level]['oov_words']:
        if verb in bloom_verbs[level]:
            level_oov_matches[level].append(verb)

for level in bloom_verbs.keys():
    print(level + ': ')
    print(level_oov_matches[level])


knowledge: 
['recite', 'draw', 'relate']
comprehension: 
['restate', 'paraphrase']
application: 
['relate', 'modify']
analysis: 
['question', 'break', 'investigate', 'divide']
evaluation: 
['critique', 'conclude', 'decide']
synthesis: 
['generate', 'invent', 'combine', 'organize', 'compose', 'choose', 'revise', 'integrate', 'predict']


In [38]:
label_oov_dict.keys()

dict_keys(['analysis', 'knowledge', 'comprehension', 'evaluation', 'synthesis', 'application'])

In [39]:
bloom_verbs.keys()

dict_keys(['knowledge', 'comprehension', 'application', 'analysis', 'evaluation', 'synthesis'])

In [25]:
label_oov_dict.keys()

dict_keys(['analysis', 'knowledge', 'comprehension', 'evaluation', 'synthesis', 'application'])

['etc',
 'express',
 'water',
 'who',
 'recite',
 'hour',
 'taking',
 'queue',
 'unique',
 'family',
 'titration',
 'usa',
 'writer',
 'memory',
 'gas',
 'flow',
 'particular',
 'movement',
 'pictoral',
 'engineer',
 'car',
 'sentence',
 'protection',
 'strike',
 'correctly',
 'rate',
 'typical',
 'movie',
 'data',
 's',
 'corrosion',
 'see',
 'paint',
 'wreck',
 'airport',
 'olympic',
 'un',
 'traceability',
 'floodplain',
 'substrate',
 'supply',
 'cornell',
 'group',
 'position',
 'pay',
 'not',
 'europe',
 'ordinal',
 'mockingbird',
 'penetration',
 'interest',
 'old',
 'season',
 'clause',
 'arabia',
 'suriname',
 'sound',
 'show',
 'elephant',
 'c',
 'arabic',
 'bear',
 'equation',
 'man',
 'processing',
 'house',
 'continent',
 'simile',
 'hypotonic',
 'artist',
 'hamilton',
 'food',
 'cubist',
 'software',
 'draw',
 'assignment',
 'loop',
 'chart',
 'dictionary',
 'insect',
 'compose',
 'microscope',
 'index',
 'volume',
 'build',
 'location',
 'implementation',
 'lobe',
 'test

In [ ]:
label_oov_dict['l']

{'analysis': {'num_oov': 242,
  'oov_words': ['paris',
   'mainstream',
   'sell',
   'water',
   'action',
   'significant',
   'attack',
   'spring',
   'marineris',
   'series',
   'heterodox',
   'family',
   'venn',
   'contemporary',
   'interact',
   'trip',
   'privilege',
   'pot',
   'interrogation',
   'switch',
   'make',
   'iris',
   'gps',
   'feeling',
   'where',
   'camera',
   'psychologist',
   'rct',
   'class',
   'fact',
   'ago',
   'operator',
   'buyer',
   'narrative',
   'pet',
   'adolescent',
   'preview',
   'safe',
   'coastal',
   'reasonably',
   'ethnic',
   'conjugal',
   'olympic',
   'floodplain',
   'place',
   'attitude',
   'society',
   'z',
   'position',
   'purge',
   'pointwise',
   'dog',
   'question',
   'heme',
   'male',
   'asd',
   'course',
   'tectonic',
   'td',
   'endorsement',
   'old',
   'gun',
   'carnivorous',
   'positivist',
   'undercover',
   'rum',
   'positive',
   'fox',
   'kuhn',
   'influential',
   'exclude',
   